# Square 1: Where does GPU acceleration actually matter?

**Goal:** sweep across data sizes and operation types (both raw data-wrangling and real ML models) —
cuDF vs pandas, cuML vs sklearn — to find where the GPU speedup is dramatic. That result *is* the
idea-selection tool: whichever operation shows the biggest, most defensible speedup points at the
kind of real-world tool worth building (forecasting → demand planning, clustering → segmentation,
classification → risk/ticket scoring, joins/groupby → dashboard/BI backend, etc).

Uses one synthetic "generic transactions" schema (store/product/user/timestamp/amount/region) that
maps to retail, support-ops, finance, or logistics equally — so results stay usable no matter which
domain you land on.

**Time budget:** RAPIDS install (~5-8 min) + sweep (~15-25 min depending on how high you push
`SIZES`). Start with the default sizes, look at results, then re-run Section 3/4 with bigger sizes
only for the operations that looked promising.

## 0. Install RAPIDS (cuDF + cuML)
Kaggle T4 runtime, CUDA 12. This is the slowest cell — kick it off first.

In [ ]:
%%capture
!pip install -q --extra-index-url=https://pypi.nvidia.com \
    cudf-cu12 cuml-cu12 cupy-cuda12x
!pip install -q scikit-learn matplotlib pandas

In [ ]:
import cudf, cuml
import pandas as pd, numpy as np
import time

print("cuDF:", cudf.__version__, "| cuML:", cuml.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Synthetic data generator (generic transactions schema)
Adjust `SIZES` to control the sweep. Start small (fast feedback), push bigger for the
operations that look promising.

In [ ]:
SIZES = [100_000, 1_000_000, 5_000_000, 20_000_000]   # row counts to sweep
N_STORES, N_PRODUCTS, N_USERS, N_REGIONS = 200, 2000, 50_000, 8

def make_transactions(n, seed=42):
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        "store_id":    rng.integers(0, N_STORES, n),
        "product_id":  rng.integers(0, N_PRODUCTS, n),
        "user_id":     rng.integers(0, N_USERS, n),
        "region":      rng.integers(0, N_REGIONS, n),
        "timestamp":   pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 365*24*3600, n), unit="s"),
        "amount":      rng.exponential(40, n).round(2),
        "qty":         rng.integers(1, 10, n),
        "discount_pct": rng.uniform(0, 0.3, n).round(3),
        "is_return":   rng.random(n) < 0.05,
        # a few numeric features for the ML sweep
        "feat_1": rng.normal(0, 1, n),
        "feat_2": rng.normal(0, 1, n),
        "feat_3": rng.normal(0, 1, n),
        "feat_4": rng.normal(0, 1, n),
    })

# target for classification (risk / "needs attention" flag) -- weakly correlated with features
def add_target(df):
    score = (df["feat_1"] * 0.6 + df["feat_2"] * -0.4 + df["amount"] / 100
             + np.random.default_rng(0).normal(0, 1, len(df)))
    df["target_flag"] = (score > score.median()).astype(int)
    df["target_value"] = score  # for regression sweep
    return df

print("Generator ready. Example (10k rows):")
add_target(make_transactions(10_000)).head(3)

## 2. Timing harness

In [ ]:
results = []  # rows: {op, size, engine, seconds}

def timeit(fn, *args, **kwargs):
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return out, time.perf_counter() - t0

def record(op, size, engine, seconds):
    results.append({"op": op, "size": size, "engine": engine, "seconds": seconds})
    print(f"  [{engine:6s}] {op:22s} n={size:>11,} -> {seconds:8.3f}s")

## 3. Data-wrangling sweep: groupby / join / sort / rolling-window (cuDF vs pandas)

In [ ]:
for n in SIZES:
    print(f"\n=== n={n:,} ===")
    pdf = add_target(make_transactions(n))
    gdf = cudf.DataFrame.from_pandas(pdf)

    # --- groupby + agg (e.g. "revenue by store x product") ---
    _, t = timeit(lambda: pdf.groupby(["store_id", "product_id"]).agg({"amount": "sum", "qty": "sum"}))
    record("groupby_agg", n, "pandas", t)
    _, t = timeit(lambda: gdf.groupby(["store_id", "product_id"]).agg({"amount": "sum", "qty": "sum"}))
    record("groupby_agg", n, "cudf", t)

    # --- merge/join (e.g. joining a small dimension table) ---
    store_dim_pd = pd.DataFrame({"store_id": range(N_STORES), "store_tier": np.random.choice(["A","B","C"], N_STORES)})
    store_dim_gd = cudf.DataFrame.from_pandas(store_dim_pd)
    _, t = timeit(lambda: pdf.merge(store_dim_pd, on="store_id", how="left"))
    record("merge_join", n, "pandas", t)
    _, t = timeit(lambda: gdf.merge(store_dim_gd, on="store_id", how="left"))
    record("merge_join", n, "cudf", t)

    # --- sort ---
    _, t = timeit(lambda: pdf.sort_values("amount"))
    record("sort", n, "pandas", t)
    _, t = timeit(lambda: gdf.sort_values("amount"))
    record("sort", n, "cudf", t)

    # --- rolling window per store (time-series feature engineering) ---
    _, t = timeit(lambda: pdf.sort_values("timestamp").groupby("store_id")["amount"].rolling(7).mean())
    record("rolling_window", n, "pandas", t)
    try:
        _, t = timeit(lambda: gdf.sort_values("timestamp").groupby("store_id")["amount"].rolling(7).mean())
        record("rolling_window", n, "cudf", t)
    except Exception as e:
        print("  [cudf] rolling_window unsupported/errored:", e)

## 4. ML sweep: forecasting / clustering / classification (cuML vs sklearn)
This is usually where the biggest, most defensible speedups show up.

In [ ]:
from sklearn.linear_model import LinearRegression as skLinearRegression
from sklearn.cluster import KMeans as skKMeans
from sklearn.ensemble import RandomForestClassifier as skRandomForestClassifier
from cuml.linear_model import LinearRegression as cuLinearRegression
from cuml.cluster import KMeans as cuKMeans
from cuml.ensemble import RandomForestClassifier as cuRandomForestClassifier

FEATURES = ["feat_1", "feat_2", "feat_3", "feat_4", "amount", "qty"]

for n in SIZES:
    print(f"\n=== n={n:,} ===")
    pdf = add_target(make_transactions(n))
    gdf = cudf.DataFrame.from_pandas(pdf)

    X_pd, y_reg_pd, y_clf_pd = pdf[FEATURES], pdf["target_value"], pdf["target_flag"]
    X_gd = gdf[FEATURES].astype("float32")
    y_reg_gd = gdf["target_value"].astype("float32")
    y_clf_gd = gdf["target_flag"].astype("int32")

    # --- Linear regression (e.g. demand/revenue forecasting) ---
    _, t = timeit(lambda: skLinearRegression().fit(X_pd, y_reg_pd))
    record("linreg_fit", n, "sklearn", t)
    _, t = timeit(lambda: cuLinearRegression().fit(X_gd, y_reg_gd))
    record("linreg_fit", n, "cuml", t)

    # --- KMeans (e.g. customer/store segmentation) ---
    _, t = timeit(lambda: skKMeans(n_clusters=8, n_init=3, random_state=0).fit(X_pd))
    record("kmeans_fit", n, "sklearn", t)
    _, t = timeit(lambda: cuKMeans(n_clusters=8, n_init=3, random_state=0).fit(X_gd))
    record("kmeans_fit", n, "cuml", t)

    # --- RandomForest classification (e.g. risk scoring / ticket triage priority) ---
    # cap sklearn RF at smaller sizes -- it gets painfully slow past a few million rows
    if n <= 5_000_000:
        _, t = timeit(lambda: skRandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1).fit(X_pd, y_clf_pd))
        record("rf_classify_fit", n, "sklearn", t)
    else:
        print(f"  [sklearn] rf_classify_fit skipped at n={n:,} (too slow on CPU -- that's the point)")
    _, t = timeit(lambda: cuRandomForestClassifier(n_estimators=50, max_depth=10).fit(X_gd, y_clf_gd))
    record("rf_classify_fit", n, "cuml", t)

## 5. Results: speedup by operation and scale

In [ ]:
res_df = pd.DataFrame(results)
res_df.to_csv("gpu_sweep_results.csv", index=False)

pivot = res_df.pivot_table(index=["op", "size"], columns="engine", values="seconds")
cpu_col = next(c for c in ["pandas", "sklearn"] if c in pivot.columns)  # per-op cpu engine varies

# build speedup table per op (cpu_engine / gpu_engine)
speedups = []
for op in res_df["op"].unique():
    sub = res_df[res_df["op"] == op]
    cpu_engine = "pandas" if "pandas" in sub["engine"].values else "sklearn"
    gpu_engine = "cudf" if "cudf" in sub["engine"].values else "cuml"
    for n in sub["size"].unique():
        cpu_t = sub[(sub["size"] == n) & (sub["engine"] == cpu_engine)]["seconds"]
        gpu_t = sub[(sub["size"] == n) & (sub["engine"] == gpu_engine)]["seconds"]
        if len(cpu_t) and len(gpu_t) and gpu_t.values[0] > 0:
            speedups.append({"op": op, "size": n, "speedup_x": cpu_t.values[0] / gpu_t.values[0]})

speedup_df = pd.DataFrame(speedups).sort_values(["op", "size"])
print("Speedup (CPU seconds / GPU seconds) by operation and scale:")
display(speedup_df.pivot(index="size", columns="op", values="speedup_x").round(1))

print("\nBiggest speedups overall (top 10 rows):")
display(speedup_df.sort_values("speedup_x", ascending=False).head(10))

In [ ]:
import matplotlib.pyplot as plt

ops = speedup_df["op"].unique()
fig, ax = plt.subplots(figsize=(10, 6))
for op in ops:
    sub = speedup_df[speedup_df["op"] == op].sort_values("size")
    ax.plot(sub["size"], sub["speedup_x"], marker="o", label=op)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("rows")
ax.set_ylabel("speedup (CPU time / GPU time)")
ax.set_title("Where does GPU acceleration actually pay off?")
ax.axhline(1, color="gray", linestyle="--", linewidth=1)
ax.legend()
plt.tight_layout()
plt.savefig("gpu_speedup_sweep.png", dpi=150)
plt.show()

## 6. Reading the result — how this picks your idea

Look at which operation has the **largest, most consistent speedup as size grows**:

| If the winner is… | …it points toward |
|---|---|
| `rf_classify_fit` / `kmeans_fit` (ML training) | A **scoring/ranking/segmentation tool**: risk scores, ticket triage priority, customer segments, fraud flags — refreshed far more often because training that used to take minutes now takes seconds (this is the **Freshness** + **Decision Quality** axis in the rubric) |
| `linreg_fit` (regression) | A **forecasting dashboard**: demand/revenue/inventory forecasting, updated per-store per-day instead of weekly |
| `groupby_agg` / `merge_join` (data wrangling) | A **BI/dashboard backend**: e.g. a Looker-style dashboard that stays responsive as the underlying table grows from thousands to tens of millions of rows — this is the **Scale** axis |
| `sort` / `rolling_window` | A **time-series / leaderboard tool**: real-time ranking or rolling-metric alerting that needs to recompute constantly |
| Nothing shows >3-5x consistently | Don't force GPU into the story — pick a Google Cloud-layer-driven idea instead (BigQuery + Looker + Gemini) and lean on the **Scale**/**Freshness** narrative there instead of raw compute speed |

Once you see the winning operation, the next step is: pick one real user (store manager / support
lead / ops analyst / something from your own life) whose actual bottleneck is *exactly* that
operation at scale, and build the 3-stage pipeline (Ingest & Clean → Analyze & Model → Visualize &
Act) around it, citing this chart as your acceleration proof.